In [1]:
import pandas as pd

from gender_association_method import associate_gender

In [ ]:
df_inferred_gender_gpt4 = pd.read_csv('data/gpt-4o-mini/generated_personas_occupation_inferred_gender_gpt-4o-mini-2024-07-18_100_12-03-2024, 15:29:10.csv')
                                    
df_gender_gpt4 = pd.read_csv('data/gpt-4o-mini/generated_personas_occupation_from_winogender_gender_gpt-4o-mini-2024-07-18_100_11-11-2024, 12:02:49.csv')

df_inferred_gender_gpt3_5 = pd.read_csv('data/gpt-3.5/generated_personas_occupation_inferred_gender_gpt-3.5-turbo-0125_100_12-06-2024, 20:25:04.csv')
df_gender_gpt3_5 = pd.read_csv('data/gpt-3.5/generated_personas_occupation_from_winogender_gender_gpt-3.5-turbo-0125_100_11-12-2024, 08:10:32.csv')

df_gender_llama3 = pd.read_csv('data/llama-3.1-70b/generated_personas_occupation_from_winogender_gender_meta-llama-Meta-Llama-3.1-70B-Instruct-Turbo_100_11-23-2024, 09:50:14.csv')
df_inferred_gender_llama3 = pd.read_csv('data/llama-3.1-70b/generated_personas_occupation_inferred_gender_meta-llama-Meta-Llama-3.1-70B-Instruct-Turbo_100_11-27-2024, 14:04:28.csv')

df_no_dem_gpt4 = pd.read_csv('data/gpt-4o-mini/generated_personas_occupation_no_demographics_gpt-4o-mini-2024-07-18_100_12-06-2024, 16:21:41.csv')
df_no_dem_gpt3 = pd.read_csv('data/gpt-3.5/generated_personas_occupation_no_demographics_gpt-3.5-turbo-0125_100_08-11-2024, 04:22:20.csv')
df_no_dem_llama3 = pd.read_csv('data/llama-3.1-70b/generated_personas_occupation_no_demographics_meta-llama-Meta-Llama-3.1-70B-Instruct-Turbo_100_11-21-2024, 07:28:21.csv')



In [4]:
def get_infer_gender_acc(df, occupations, num_gens=100):
    counts_by_gender = dict()
    not_captured = dict()
    
    for occupation in occupations:
        prompts = df[df['occupation'] == occupation]['prompt'].unique()

        counts_by_gender[occupation] = dict()
        not_captured[occupation] = dict()
        for prompt in prompts:
            counts_by_gender[occupation][prompt] = {'M': 0, 'F': 0, 'N': 0}
            not_captured[occupation][prompt] = 0
            prompt_df = df[df['prompt'] == prompt]
            sampled_df = prompt_df.sample(n=num_gens, random_state=42)

            for i in range(len(sampled_df['text'])):
                g = None
                
                if type(sampled_df['text'].iloc[i]) != str:
                    continue

                text = sampled_df['text'].iloc[i].lower()
                g = associate_gender(text)
                if g:
                    counts_by_gender[occupation][prompt][g] += 1
                else:
                    not_captured[occupation][prompt] += 1

    return counts_by_gender, not_captured
        

# Calculates the percent captured of generations where gender is unknown

In [6]:
occupation_stats_filename = 'occupations_stats_from_winogender.tsv'
occupation_data=pd.read_csv(occupation_stats_filename,sep='\t')

occupations = occupation_data['occupation'].unique()
counts_by_gender_gpt4_new, not_captured_gpt4_new = get_infer_gender_acc(df_no_dem_gpt4, occupations)

counts_by_gender_gpt3_new, not_captured_gpt3_new = get_infer_gender_acc(df_no_dem_gpt3, occupations)

counts_by_gender_llama3_new, not_captured_llama3_new = get_infer_gender_acc(df_no_dem_llama3, occupations)


In [35]:
def calc_sum(counts, gender=None):
    total_sum = 0
    for occ in counts.keys():
        for prompt in counts[occ].keys():
            if type(counts[occ][prompt]) is dict:
                if gender:
                    total_sum += counts[occ][prompt][gender]
                else:
                    for g in counts[occ][prompt].keys():
                        total_sum += counts[occ][prompt][g]
            elif type(counts[occ][prompt]) is int:
                total_sum += counts[occ][prompt]
    return total_sum

In [13]:
print('& GPT-3.5 & GPT-4o-mini & Llama-3.1-70b\\\\')
print('\\hline')
print(f'\\% Captured & {calc_sum(counts_by_gender_gpt3_new)/(calc_sum(counts_by_gender_gpt3_new)+calc_sum(not_captured_gpt3_new))*100:.3f} & {calc_sum(counts_by_gender_gpt4_new)/(calc_sum(counts_by_gender_gpt4_new)+calc_sum(not_captured_gpt4_new))*100:.3f} & {calc_sum(counts_by_gender_llama3_new)/(calc_sum(counts_by_gender_llama3_new)+calc_sum(not_captured_llama3_new))*100:.3f}')

& GPT-3.5 & GPT-4o-mini & Llama-3.1-70b\\
\hline
\% Captured & 80.460 & 94.310 & 98.167


In [28]:
def get_gender_acc(df, occupations, num_gens=100):
    correct = dict()
    incorrect = dict()
    not_captured = dict()
    
    for occupation in occupations:
        prompts = df[df['occupation'] == occupation]['prompt'].unique()

        correct[occupation] = dict()
        incorrect[occupation] = dict()
        not_captured[occupation] = dict()
        for prompt in prompts:
            correct[occupation][prompt] = {'M': 0, 'F': 0, 'N': 0}
            incorrect[occupation][prompt] = {'M': 0, 'F': 0, 'N': 0}
            not_captured[occupation][prompt] = {'M': 0, 'F': 0, 'N': 0}
            prompt_df = df[df['prompt'] == prompt]
            sampled_df = prompt_df.sample(n=num_gens, random_state=42)

            for i in range(len(sampled_df['text'])):
                associated_g = None
                g = sampled_df['gender'].iloc[i]
                
                if type(sampled_df['text'].iloc[i]) != str:
                    continue

                text = sampled_df['text'].iloc[i].lower()
                associated_g = associate_gender(text)
                if associated_g:
                    if g == associated_g:
                        correct[occupation][prompt][g] += 1
                    else:
                        incorrect[occupation][prompt][g] += 1
                else:
                    not_captured[occupation][prompt][g] += 1

    return correct, incorrect, not_captured
        

# Calculate the accuracy of the Gender Association Method using generations where gender is known

In [29]:
correct_by_gender_gpt4_new, incorrect_by_gender_gpt4_new, not_captured_gpt4_new = get_gender_acc(df_gender_gpt4, occupations)

correct_by_gender_gpt3_new, incorrect_by_gender_gpt3_new, not_captured_gpt3_new = get_gender_acc(df_gender_gpt3_5, occupations)

correct_by_gender_llama3_new, incorrect_by_gender_llama3_new, not_captured_llama3_new = get_gender_acc(df_gender_llama3, occupations)


In [36]:
totals = {'F': 0, 'M': 0, 'N': 0}
for g in totals.keys():
    totals[g] = calc_sum(correct_by_gender_gpt3_new, g) + calc_sum(incorrect_by_gender_gpt3_new, g) + calc_sum(not_captured_gpt3_new, g) + calc_sum(correct_by_gender_gpt4_new, g) + calc_sum(incorrect_by_gender_gpt4_new, g) + calc_sum(not_captured_gpt4_new, g) + calc_sum(correct_by_gender_llama3_new, g) + calc_sum(incorrect_by_gender_llama3_new, g) + calc_sum(not_captured_llama3_new, g) 
print('Gender & Correct \\% & Incorrect\\% & Not Captured\\%\\\\')
print('\\hline')
print(f"Female & {(calc_sum(correct_by_gender_gpt3_new, 'F')+calc_sum(correct_by_gender_gpt4_new, 'F')+calc_sum(correct_by_gender_llama3_new, 'F'))/totals['F']*100:.4f} & {(calc_sum(incorrect_by_gender_gpt3_new, 'F')+calc_sum(incorrect_by_gender_gpt4_new, 'F')+calc_sum(incorrect_by_gender_llama3_new, 'F'))/totals['F']*100:.4f} & {(calc_sum(not_captured_gpt3_new, 'F')+calc_sum(not_captured_gpt4_new, 'F')+calc_sum(not_captured_llama3_new, 'F'))/totals['F']*100:.4f}\\\\")
print(f"Male & {(calc_sum(correct_by_gender_gpt3_new, 'M')+calc_sum(correct_by_gender_gpt4_new, 'M')+calc_sum(correct_by_gender_llama3_new, 'M'))/totals['M']*100:.4f} & {(calc_sum(incorrect_by_gender_gpt3_new, 'M')+calc_sum(incorrect_by_gender_gpt4_new, 'M')+calc_sum(incorrect_by_gender_llama3_new, 'M'))/totals['M']*100:.4f} & {(calc_sum(not_captured_gpt3_new, 'M')+calc_sum(not_captured_gpt4_new, 'M')+calc_sum(not_captured_llama3_new, 'M'))/totals['M']*100:.4f}\\\\")
print(f"Non-binary & {(calc_sum(correct_by_gender_gpt3_new, 'N')+calc_sum(correct_by_gender_gpt4_new, 'N')+calc_sum(correct_by_gender_llama3_new, 'N'))/totals['N']*100:.4f} & {(calc_sum(incorrect_by_gender_gpt3_new, 'N')+calc_sum(incorrect_by_gender_gpt4_new, 'N')+calc_sum(incorrect_by_gender_llama3_new, 'N'))/totals['N']*100:.4f} & {(calc_sum(not_captured_gpt3_new, 'N')+calc_sum(not_captured_gpt4_new, 'N')+calc_sum(not_captured_llama3_new, 'N'))/totals['N']*100:.4f}\\\\")

Gender & Correct \% & Incorrect\% & Not Captured\%\\
\hline
Female & 99.9180 & 0.0079 & 0.0741\\
Male & 99.8466 & 0.0053 & 0.1481\\
Non-binary & 99.6693 & 0.0026 & 0.3280\\
